
# V-JEPA Crop Dataloader Smoke Test

Quick notebook to sanity-check the crop-enabled RMMDataset. It pulls one clip, applies SAM3-based cropping (using `video_meta.json` + `rmm_sam3_parsed.csv`), and visualizes sampled frames with metadata.


In [ ]:
%matplotlib inline

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from finetune_sails_vjepa2_cv_crop import (
    RMMDataset,
    load_split,
    build_video_label_counts,
    get_frames_per_clip,
    CropConfig,
    load_parsed_sam3_csv,
    load_video_meta_json,
    VJEPA2VideoProcessor,
)

plt.rcParams["figure.figsize"] = (14, 6)

In [ ]:

# Paths / config (adjust as needed)
csv_dir = Path('/orcd/data/satra/001/users/brukew/actreg/dataprep/cv_folds')
clips_root = Path('/orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips')
sam3_parsed_csv = Path('/orcd/data/satra/001/users/brukew/actreg/dataprep/rmm_sam3_parsed.csv')
video_meta_json = Path('/orcd/data/satra/001/users/brukew/actreg/dataprep/video_meta.json')
model_id = 'facebook/vjepa2-vitl-fpc16-256-ssv2'
frames_override = 16  # set None or <=0 to use processor default

train_csvs = sorted(csv_dir.glob('fold_*_train.csv'))
assert train_csvs, 'No train CSVs found'
train_csv = train_csvs[0]
print('Using CSV:', train_csv)


In [ ]:

# Load records and label mappings
records, missing = load_split([train_csv], clips_root)
print(f"Records: {len(records)} | Missing clips: {len(missing)}")
if missing[:5]:
    print('Example missing:', missing[:5])

video_label_counts = build_video_label_counts(records)
labels = sorted({r['label'] for r in records})
label2id = {lbl: i for i, lbl in enumerate(labels)}
id2label = {i: lbl for lbl, i in label2id.items()}


In [ ]:

# Load crop resources
sam3_rows = load_parsed_sam3_csv(sam3_parsed_csv)
video_meta = load_video_meta_json(video_meta_json)
print(f"SAM3 rows: {len(sam3_rows)} | video_meta entries: {len(video_meta)}")


In [ ]:

# Build dataset (subset for speed) and pull one sample
processor = VJEPA2VideoProcessor.from_pretrained(model_id)
frames_per_clip = get_frames_per_clip(processor, frames_override)
print('Frames per clip:', frames_per_clip)

crop_cfg = CropConfig(
    enabled=True,
    sam3_parsed_csv=sam3_parsed_csv,
    mask_cache_base=Path('/orcd/scratch/bcs/001/sensein/sails/cache_for_tracking'),
    mask_model='facebook-sam3',
    mask_prompt='person',
    padding=20,
    rotation_override=None,
    fallback='full',
    video_meta_json=video_meta_json,
)

dataset = RMMDataset(records[50:70], label2id, frames_per_clip, video_label_counts, crop_cfg=crop_cfg, sam3_rows=sam3_rows, video_meta=video_meta)

sample = None
for i in range(len(dataset)):
    sample = dataset[i]
    if sample is not None:
        break

if sample is None:
    raise RuntimeError('No sample available (all skipped).')

frames_np, label_id, meta = sample
print('Frames shape:', frames_np.shape)
print('Label:', label_id, id2label[label_id])
print('Crop meta:', json.dumps(meta, indent=2))


In [ ]:
sample = dataset[19]
frames_np, label_id, meta = sample
print('Frames shape:', frames_np.shape)
print('Label:', label_id, id2label[label_id])
print('Crop meta:', json.dumps(meta, indent=2))

In [ ]:
# Visualize a few frames (and save grid to disk)
import numpy as np
from IPython.display import Image, display

# Ensure frames_np exists
try:
    _frames = frames_np
except NameError:
    raise RuntimeError('frames_np not defined; run the previous cell first')

imgs = _frames
if isinstance(imgs, torch.Tensor):
    imgs = imgs.cpu().numpy()
if imgs.dtype != np.uint8:
    imgs = np.clip(imgs, 0, 255).astype(np.uint8)

num_show = min(8, imgs.shape[0])
fig, axes = plt.subplots(1, num_show, figsize=(3*num_show, 4))
for i in range(num_show):
    axes[i].imshow(imgs[i])
    axes[i].set_title(f"Frame {i}")
    axes[i].axis('off')
plt.tight_layout()

out_png = 'crop_preview.png'
fig.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved preview to {out_png}')
try:
    display(Image(out_png))
except Exception:
    pass


In [ ]:
import numpy as np
import torch

# Ensure we actually have a sample
try:
    print("frames_np type/shape:", type(frames_np), getattr(frames_np, "shape", None))
except NameError:
    raise RuntimeError("frames_np not defined; rerun the sample cell first")

# Normalize to uint8 HWC numpy
if isinstance(frames_np, torch.Tensor):
    frames_np = frames_np.cpu().numpy()
frames_show = frames_np
if frames_show.ndim == 4 and frames_show.shape[1] in (3, 1) and frames_show.shape[-1] != 3:
    # NCHW -> NHWC
    frames_show = np.transpose(frames_show, (0, 2, 3, 1))
frames_show = frames_show.astype(np.uint8)

num_show = min(8, frames_show.shape[0])
if num_show == 0:
    raise RuntimeError("No frames to show (frames_show is empty)")

fig, axes = plt.subplots(1, num_show, figsize=(3*num_show, 4))
for i in range(num_show):
    axes[i].imshow(frames_show[i])
    axes[i].set_title(f"Frame {i}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np, matplotlib.pyplot as plt

print("dtype/min/max:", frames_np.dtype, frames_np.min(), frames_np.max())

imgs = frames_np.astype(np.uint8) if frames_np.dtype != np.uint8 else frames_np
num_show = min(8, len(imgs))
fig, axes = plt.subplots(1, num_show, figsize=(3*num_show, 4))
for i in range(num_show):
    axes[i].imshow(imgs[i])
    axes[i].set_title(f"Frame {i}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import h5py, os

cache_path = Path("/orcd/scratch/bcs/001/sensein/sails/cache_for_tracking/masks/20210912_214928_segmented/facebook-sam3__prompt-person.h5")
print("Exists:", cache_path.exists(), "size bytes:", os.path.getsize(cache_path) if cache_path.exists() else 0)

try:
    with h5py.File(cache_path, "r") as f:
        frame_keys = [k for k in f.keys() if k.startswith("frame_")]
        print("frames:", len(frame_keys), "attrs:", dict(f.attrs))
        # peek one frame group if present
        if frame_keys:
            k = frame_keys[0]
            print(k, "boxes shape:", f[k]["boxes"].shape, "obj_ids shape:", f[k]["obj_ids"].shape)
except Exception as e:
    print("Failed to read cache:", e)


In [ ]:
from decord import VideoReader, cpu
from subprocess import run

clip = "/orcd/scratch/bcs/001/sensein/sails/rmm/vjepa2_finetune_clips/fold_0_val/L4K0P3T4Q8_36_month_185_0_0.mp4"
print("Exists:", Path(clip).exists(), "size:", Path(clip).stat().st_size if Path(clip).exists() else 0)

try:
    vr = VideoReader(clip, ctx=cpu(0), fault_tol=1)
    print("num frames:", len(vr))
    _ = vr[0]
    print("First frame read OK")
except Exception as e:
    print("Decord error:", e)

# ffprobe check
run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=width,height,nb_frames", "-of", "csv=p=0", clip])
